# Polyp Detection — Safety-First Training

This notebook evaluates the baseline model at multiple confidence
thresholds rather than retraining, after discovering that YOLOv8
silently ignores the cls_pw parameter (confirmed in notebook 03
training logs: cls_pw printed as 0.0 regardless of what was passed).

The safety-first goal — higher recall at the cost of some precision —
is better achieved by tuning the inference threshold than by changing
the loss function.

---

## What this notebook does

1. Sweeps confidence thresholds from 0.50 down to 0.10 on CVC-ClinicDB
2. Runs the same sweep on Kvasir-SEG test set for comparison
3. Selects the threshold where recall plateaus (avoids unnecessary
   false positives from going lower than needed)
4. Saves all results for the final comparison in notebook 05

## Output

- results/metrics/threshold_sweep.json
- results/metrics/safety_first_metrics.json
- results/figures/threshold_sweep.png

In [ ]:
# Import libraries

import json
import os
import sys

import matplotlib.pyplot as plt
import torch
import yaml
from ultralytics import YOLO

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    "dataset_yaml":     os.path.join(BASE_DIR, "configs", "dataset.yaml"),
    "cvc_eval_yaml":    os.path.join(BASE_DIR, "configs", "dataset_cvc_eval.yaml"),
    "baseline_weights": os.path.join(BASE_DIR, "models", "baseline", "weights", "best.pt"),
    "metrics":          os.path.join(BASE_DIR, "results", "metrics"),
    "figures":          os.path.join(BASE_DIR, "results", "figures"),
}

os.makedirs(PATHS["metrics"], exist_ok=True)
os.makedirs(PATHS["figures"], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:16s} -> {path}")

In [ ]:
# Load Baseline Model
# Load the baseline model trained in notebook 03

print(f"Loading baseline model: {PATHS['baseline_weights']}")
model = YOLO(PATHS["baseline_weights"])
print("Model loaded successfully.")

In [ ]:
# Threshold Sweep Function
# Sweep across multiple confidence thresholds and record
# recall/precision at each one on the cross-dataset test set

def threshold_sweep(model, data_yaml, split, thresholds):
    sweep_results = []

    for conf in thresholds:
        metrics = model.val(
            data=data_yaml,
            split=split,
            imgsz=640,
            conf=conf,
            verbose=False,
        )
        sweep_results.append({
            "threshold": conf,
            "precision": float(metrics.box.mp),
            "recall":    float(metrics.box.mr),
            "map50":     float(metrics.box.map50),
        })
        print(f"  conf={conf:.2f}  ->  "
              f"Recall: {metrics.box.mr:.3f}  "
              f"Precision: {metrics.box.mp:.3f}  "
              f"mAP50: {metrics.box.map50:.3f}")

    return sweep_results


THRESHOLDS = [0.50, 0.40, 0.30, 0.25, 0.20, 0.15, 0.10]

In [ ]:
# Run Sweep on CVC-ClinicDB
# This is where the recall problem was worst (75.2% at conf=0.50)

print("BASELINE MODEL - CVC-ClinicDB - Threshold Sweep")
print("-" * 50)

cvc_sweep = threshold_sweep(
    model, PATHS["cvc_eval_yaml"], "val", THRESHOLDS
)

In [ ]:
# Run Sweep on Kvasir-SEG Test
# Same sweep on the in-distribution test set, so we can see
# whether lowering the threshold hurts in-distribution precision
# more than it helps cross-dataset recall

print("BASELINE MODEL - Kvasir-SEG Test - Threshold Sweep")
print("-" * 50)

kvasir_sweep = threshold_sweep(
    model, PATHS["dataset_yaml"], "test", THRESHOLDS
)

In [ ]:
# Plot Threshold Sweep
# Visualize recall and precision across thresholds for both
# datasets. This is the core evidence for the safety-first story.

cvc_recalls       = [r["recall"] for r in cvc_sweep]
cvc_precisions    = [r["precision"] for r in cvc_sweep]
kvasir_recalls    = [r["recall"] for r in kvasir_sweep]
kvasir_precisions = [r["precision"] for r in kvasir_sweep]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Recall vs threshold
axes[0].plot(THRESHOLDS, kvasir_recalls, marker="o",
             color="steelblue", label="Kvasir-SEG (in-distribution)")
axes[0].plot(THRESHOLDS, cvc_recalls, marker="o",
             color="coral", label="CVC-ClinicDB (cross-dataset)")
axes[0].axhline(0.93, color="red", linestyle="--", linewidth=1,
                label="Target recall (93%)")
axes[0].set_xlabel("Confidence threshold")
axes[0].set_ylabel("Recall")
axes[0].set_title("Recall vs Threshold")
axes[0].invert_xaxis()
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision vs threshold
axes[1].plot(THRESHOLDS, kvasir_precisions, marker="o",
             color="steelblue", label="Kvasir-SEG (in-distribution)")
axes[1].plot(THRESHOLDS, cvc_precisions, marker="o",
             color="coral", label="CVC-ClinicDB (cross-dataset)")
axes[1].set_xlabel("Confidence threshold")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision vs Threshold")
axes[1].invert_xaxis()
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "threshold_sweep.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Select Best Threshold
# Pick the highest threshold (most conservative) that still
# achieves recall > 0.93 on the cross-dataset test set.
# We optimize for CVC-ClinicDB specifically, since that's the
# realistic "new hospital" scenario this whole project targets.

TARGET_RECALL = 0.93

candidates = [r for r in cvc_sweep if r["recall"] >= TARGET_RECALL]

if candidates:
    best = max(candidates, key=lambda r: r["precision"])
    print(f"Selected threshold: {best['threshold']:.2f}")
    print(f"  Recall:    {best['recall']:.3f}")
    print(f"  Precision: {best['precision']:.3f}")
    print(f"  mAP50:     {best['map50']:.3f}")
else:
    best = max(cvc_sweep, key=lambda r: r["recall"])
    print(f"WARNING: no threshold reached {TARGET_RECALL:.0%} recall.")
    print(f"Best available threshold: {best['threshold']:.2f}")
    print(f"  Recall:    {best['recall']:.3f}")
    print(f"  Precision: {best['precision']:.3f}")

SELECTED_THRESHOLD = best["threshold"]

In [ ]:
# Final Results at Selected Threshold
# Pull the already-computed sweep results at the selected
# threshold for both datasets - no need to re-run val()

kvasir_at_selected = next(
    r for r in kvasir_sweep if r["threshold"] == SELECTED_THRESHOLD
)
cvc_at_selected = next(
    r for r in cvc_sweep if r["threshold"] == SELECTED_THRESHOLD
)

print(f"FINAL RESULTS AT SELECTED THRESHOLD ({SELECTED_THRESHOLD:.2f})")
print("=" * 50)
print("\nKvasir-SEG test set:")
print(f"  Recall:    {kvasir_at_selected['recall']:.4f}")
print(f"  Precision: {kvasir_at_selected['precision']:.4f}")
print(f"  mAP50:     {kvasir_at_selected['map50']:.4f}")

print("\nCVC-ClinicDB cross-dataset:")
print(f"  Recall:    {cvc_at_selected['recall']:.4f}")
print(f"  Precision: {cvc_at_selected['precision']:.4f}")
print(f"  mAP50:     {cvc_at_selected['map50']:.4f}")

In [ ]:
# Save All Metrics
# Save everything needed for notebook 05's final comparison

safety_first_metrics = {
    "model": "YOLOv8m-seg (same weights as baseline)",
    "config": f"safety-first (threshold={SELECTED_THRESHOLD:.2f}, no retraining)",
    "selected_threshold": SELECTED_THRESHOLD,
    "kvasir_test": kvasir_at_selected,
    "cvc_cross_dataset": cvc_at_selected,
}

metrics_path = os.path.join(PATHS["metrics"], "safety_first_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(safety_first_metrics, f, indent=2)
print(f"Saved -> {metrics_path}")

sweep_data = {
    "thresholds": THRESHOLDS,
    "kvasir_sweep": kvasir_sweep,
    "cvc_sweep":    cvc_sweep,
}

sweep_path = os.path.join(PATHS["metrics"], "threshold_sweep.json")
with open(sweep_path, "w") as f:
    json.dump(sweep_data, f, indent=2)
print(f"Saved -> {sweep_path}")

In [ ]:
# Summary

print("NOTEBOOK 04 COMPLETE")
print("=" * 50)
print(f"Selected threshold: {SELECTED_THRESHOLD:.2f}")
print()
print("CVC-ClinicDB cross-dataset comparison:")
print(f"  Default  (conf=0.50): Recall={cvc_sweep[0]['recall']:.3f}, "
      f"Precision={cvc_sweep[0]['precision']:.3f}")
print(f"  Safety-First (conf={SELECTED_THRESHOLD:.2f}): "
      f"Recall={cvc_at_selected['recall']:.3f}, "
      f"Precision={cvc_at_selected['precision']:.3f}")
print()
print("Next -> 05_evaluation.ipynb")
print("  - Build final comparison tables and visualizations")
print("  - Generate confusion matrices and error analysis")
print("  - Write results summary for README")